# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Appendix : **
##### Version Number: 3.0
---
### Contents  
> 1. ** 
> 2. **
---
### Notes

- 
---
### Inputs
- 
---
### Outputs  
- 
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
import pandas as pd
import pygeoprocessing

In [3]:
from IPython.display import display, HTML

display(HTML("""
<style>
.dataframe {
    max-height: none !important;
    overflow: visible !important;
}
</style>
"""))

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)

In [5]:
layer = pd.read_csv('../data/raw/Layers.csv')
ops = pd.read_csv('../data/raw/Operations.csv')

In [6]:
layer.columns

Index(['layer_id', 'layer_layer_name', 'layer_original', 'layer_source_path',
       'layer_source_URL', 'layer_file_type', 'geometry_type',
       'layer_category', 'layer_crs', 'layer_description', 'layer_notes',
       'layer_status'],
      dtype='object')

In [7]:
ops['input_names'] = [[] for _ in range(len(ops))]
ops['output_names'] = [[] for _ in range(len(ops))]
layer['operation'] = [[] for _ in range(len(layer))]

In [8]:
inputs = []
for index, row in ops.iterrows():
    in_split = row['input_ids'].split(',')
    out_split = row['output_ids'].split(',')
    for i in range(len(in_split)):
        copy = layer[layer['layer_id'] == int(in_split[i])]
        in_layer_name = copy['layer_layer_name'].iloc[0]
        row['input_names'].append(in_layer_name)
        
    for i in range(len(out_split)):
        copy = layer[layer['layer_id'] == int(out_split[i])]
        out_layer_name = copy['layer_layer_name'].iloc[0]
        row['output_names'].append(out_layer_name) 

In [9]:
ops['tool_used'] = ops['operation_toolbox'] +" -> " + ops['operation_toolset'] + " -> " + ops['operation_tool']

In [10]:
ops = ops.replace('na','')

In [11]:
ops = ops.fillna('')

In [12]:
ops = ops.drop(columns = ['operation_toolbox','operation_toolset','operation_tool','input_ids','output_ids'])

In [13]:
final_order = ['executive_operation','operation_id','operation_title','input_names','tool_used','operation_settings','output_names','operation_notes']

In [14]:
ops = ops[final_order]

In [15]:
layer = layer.replace({'na':''})
layer = layer.fillna('')
layer = layer.drop(columns = 'operation')

In [16]:
layer.columns

Index(['layer_id', 'layer_layer_name', 'layer_original', 'layer_source_path',
       'layer_source_URL', 'layer_file_type', 'geometry_type',
       'layer_category', 'layer_crs', 'layer_description', 'layer_notes',
       'layer_status'],
      dtype='object')

In [17]:
layer_order = ['layer_id', 'layer_layer_name','layer_file_type', 'geometry_type', 'layer_crs',
       'layer_category', 'layer_description', 'layer_notes','layer_status','layer_source_path', 'layer_source_URL',
        'layer_original']

In [18]:
layer = layer[layer_order]

In [19]:
layer

,layer_id,layer_layer_name,layer_file_type,geometry_type,layer_crs,layer_category,layer_description,layer_notes,layer_status,layer_source_path,layer_source_URL,layer_original
0,1,CA_Counties,Shape,Polygon,4326,Geographic,boundary file for California,Needs reprojecting,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\ca_counties\CA_Counties.shp,https://data.ca.gov/dataset/ca-geographic-boundaries,CA_Counties.shp
1,2,California_state,Shape,Polygon,3310,Grid,Reprojected operational California shapefile,,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\California_counties,,
2,3,Sampling_grid_unclipped,Feature,Polygon,3310,Grid,An equally spaced grid covering California,Needs to be clipped,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\Sampling_grid_unclipped,,
3,4,Sampling_points_label,Feature,Point,3310,Grid,Contains the centroids for 46 Km each grid,Needs to be clipped,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\Sampling_points_label,,
4,5,Sampling_grid,Feature,Polygon,3310,Grid,An equally spaced grid clipped to California,Some grids extend outside the boundaries of the state. Metrics may need to be normalized by area.,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\Sampling_grid,,
5,6,Sampling_grid_in_cali,Feature,Polygon,3310,Grid,An equally spaced grid clipped to California,Some grids extend outside the boundaries of the state. Metrics may need to be normalized by area.,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\Sampling_grid_in_cali,,
6,7,Sampling_grid_area,Feature,Polygon,3310,Grid,sampling grid with area in california field added,,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\Sampling_grid_area,,
7,8,California_state,Feature,Polygon,3310,Grid,State boundary only,,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\California_state,,
8,9,Wildland_Urban_Interface,Shape,Polygon,3310,Social,,,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Wildland_Urban_Interface\Wildland_Urban_Interface.shp,https://gis.data.ca.gov/datasets/CALFIRE-Forestry::wildland-urban-interface/explore?location=34.403601%2C-118.894358%2C9.95,Wildland_Urban_Interface
9,10,Sampling_grid_WUI_zones,Feature,Polygon,3310,Social,This layer is currently unused.,,Permanent,C:\Users\dusti\Desktop\Cal_Fire_Arc\Cal_Fire_Arc.gdb\Sampling_grid_WUI,,


In [20]:
ops.to_csv('../data/raw/ops.csv',index = False)
layer.to_csv('../data/raw/lay.csv', index = False)
print("All datasets saved successfully.")

All datasets saved successfully.
